# Trabalho Final - Inteligência Computacional

Este notebook apresenta todos os experimentos de regressão e classificação multiclasse conforme solicitado.

## Estrutura
- Importação de bibliotecas
- Carregamento dos dados
    - Regressão
        - Energy Efficiency Dataset
        - California Housing
    - Classificação
        - Wine Quality Dataset
        - Iris
- Pré-processamento
    - Regressão
        - Energy Efficiency Dataset
        - California Housing
    - Classificação
        - Wine Quality Dataset
        - Iris
- Configuração dos algoritmos
    - Regressão:
        - Regressão Linear
        - Random Forest Regressor
        - MLPRegressor (Rede Neural)
        - SVR (Support Vector Regressor)
        - FuzzyRegressor (exemplo simples usando lógica fuzzy)
    - Classificação:
        - Regressão Logística (LogisticRegression)
        - Random Forest Classifier
        - MLPClassifier (Rede Neural Multicamadas)
        - SVM Classifier (Support Vector Machine)
- Execução dos experimentos
    - Experimento 01 - Regressão
        - Setup 1 - Energy Efficiency Dataset
            - Regressão Linear
            - Random Forest Regressor
            - MLPRegressor (Rede Neural)
            - SVR (Support Vector Regressor)
            - FuzzyRegressor (exemplo simples usando lógica fuzzy)
        - Setup 2 - California Housing
            - Regressão Linear
            - Random Forest Regressor
            - MLPRegressor (Rede Neural)
            - SVR (Support Vector Regressor)
            - FuzzyRegressor (exemplo simples usando lógica fuzzy)
    - Experimento 02 - Classificação Multiclasse
        - Setup 1 - Wine Quality Dataset
            - Regressão Logística (LogisticRegression)
            - Random Forest Classifier
            - MLPClassifier (Rede Neural Multicamadas)
            - SVM Classifier (Support Vector Machine)
        - Setup 2 - Iris
            - Regressão Logística (LogisticRegression)
            - Random Forest Classifier
            - MLPClassifier (Rede Neural Multicamadas)
            - SVM Classifier (Support Vector Machine)
- Análise dos resultados
- Justificativas e análise crítica

In [1]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, f1_score, cohen_kappa_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.svm import SVR, SVC
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'pandas'

## Carregamento dos Dados
Serão utilizados os seguintes datasets:
- Regressão: `energy_efficiency.csv`, `california_housing.csv`
- Classificação: `winequality_red.csv`, `iris.csv`
Os arquivos estão na pasta `datasets_salvos/`.

In [ ]:
# Carregando os datasets
energy = pd.read_csv('datasets_salvos/energy_efficiency.csv')
california = pd.read_csv('datasets_salvos/california_housing.csv')
wine = pd.read_csv('datasets_salvos/winequality_red.csv')
iris = pd.read_csv('datasets_salvos/iris.csv')
print('Energy Efficiency Dataset:')
display(energy.head())
print('California Housing Dataset:')
display(california.head())
print('Wine Quality Red Dataset:')
display(wine.head())
print('Iris Dataset:')
display(iris.head())

## Pré-processamento dos Dados
Serão aplicadas as etapas de tratamento de valores faltantes, normalização/padronização e codificação de variáveis categóricas.

In [ ]:
# Funções de pré-processamento
def preprocess_regression(df, target_cols):
    df = df.dropna()
    X = df.drop(columns=target_cols)
    y = df[target_cols]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y
def preprocess_classification(df, target_col):
    df = df.dropna()
    X = df.drop(columns=[target_col])
    y = df[target_col]
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y
# Pré-processamento
X_energy, y_energy = preprocess_regression(energy, ['Y1', 'Y2'])
X_california, y_california = preprocess_regression(california, ['MedHouseVal'])
X_wine, y_wine = preprocess_classification(wine, 'quality')
X_iris, y_iris = preprocess_classification(iris, 'target')

## Configuração dos Algoritmos
Serão utilizados os seguintes modelos para cada tarefa:

In [ ]:
# Funções de avaliação
def regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2}
def classification_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')
    kappa = cohen_kappa_score(y_true, y_pred)
    return {'Acuracia': acc, 'F1-score': f1, 'Kappa': kappa}
# Modelos de regressão
def fuzzy_regression(X, y):
    X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0) + 1e-8)
    y_pred = np.dot(X_norm, np.ones(X_norm.shape[1]) / X_norm.shape[1])
    y_pred = y_pred * (y.max() - y.min()) + y.min()
    return y_pred
regression_models = {
    'LinearRegression': LinearRegression,
    'RandomForestRegressor': lambda: RandomForestRegressor(n_estimators=100),
    'MLPRegressor': lambda: MLPRegressor(max_iter=500),
    'SVR': lambda: SVR(kernel='rbf'),
    'FuzzyRegressor': 'fuzzy'
}
# Modelos de classificação
classification_models = {
    'LogisticRegression': LogisticRegression,
    'RandomForestClassifier': lambda: RandomForestClassifier(n_estimators=100),
    'MLPClassifier': lambda: MLPClassifier(max_iter=500),
    'SVMClassifier': lambda: SVC(kernel='rbf', probability=True)
}

## Execução dos Experimentos
Serão realizados experimentos de regressão e classificação com 30 repetições para cada modelo e base.

In [ ]:
# Funções de execução dos experimentos
def run_regression_experiment(X, y, model, seeds=range(1,31), test_size=0.3):
    results = []
    for seed in seeds:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)
        if model == 'fuzzy':
            y_pred = fuzzy_regression(X_test, y_test)
        else:
            mdl = model() if callable(model) else model
            mdl.fit(X_train, y_train)
            y_pred = mdl.predict(X_test)
        metrics = regression_metrics(y_test, y_pred)
        results.append(metrics)
    return pd.DataFrame(results)
def run_classification_experiment(X, y, model, seeds=range(1,31), test_size=0.2):
    results = []
    for seed in seeds:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
        mdl = model() if callable(model) else model
        mdl.fit(X_train, y_train)
        y_pred = mdl.predict(X_test)
        metrics = classification_metrics(y_test, y_pred)
        results.append(metrics)
    return pd.DataFrame(results)

### Experimento 01 - Regressão
Setup 1 - Energy Efficiency Dataset
Setup 2 - California Housing

In [ ]:
# Experimentos de Regressão
regression_results = {}
for name, model in regression_models.items():
    # Energy Efficiency Y1
    res = run_regression_experiment(X_energy, y_energy['Y1'], model)
    regression_results[f'Energy_{name}_Y1'] = res
    print(f'\nModelo: {name} (Energy Y1)')
    print(res.describe())
    # Energy Efficiency Y2
    res = run_regression_experiment(X_energy, y_energy['Y2'], model)
    regression_results[f'Energy_{name}_Y2'] = res
    print(f'\nModelo: {name} (Energy Y2)')
    print(res.describe())
    # California Housing
    res = run_regression_experiment(X_california, y_california, model)
    regression_results[f'California_{name}'] = res
    print(f'\nModelo: {name} (California Housing)')
    print(res.describe())

### Experimento 02 - Classificação Multiclasse
Setup 1 - Wine Quality Dataset
Setup 2 - Iris

In [ ]:
# Experimentos de Classificação
classification_results = {}
for name, model in classification_models.items():
    # Wine Quality
    res = run_classification_experiment(X_wine, y_wine, model)
    classification_results[f'Wine_{name}'] = res
    print(f'\nModelo: {name} (Wine Quality)')
    print(res.describe())
    # Iris
    res = run_classification_experiment(X_iris, y_iris, model)
    classification_results[f'Iris_{name}'] = res
    print(f'\nModelo: {name} (Iris)')
    print(res.describe())

## Análise dos Resultados
Serão apresentados gráficos comparativos das métricas dos modelos para cada base de dados.

In [ ]:
# Gráficos comparativos das métricas dos modelos
def plot_metric_comparison(results_dict, metric, title):
    plt.figure(figsize=(10,6))
    for key, df in results_dict.items():
        if metric in df.columns:
            plt.bar(key, df[metric].mean())
    plt.ylabel(metric)
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()
# Gráficos de regressão
plot_metric_comparison(regression_results, 'RMSE', 'Comparação RMSE - Regressão')
plot_metric_comparison(regression_results, 'MAE', 'Comparação MAE - Regressão')
plot_metric_comparison(regression_results, 'R2', 'Comparação R² - Regressão')
# Gráficos de classificação
plot_metric_comparison(classification_results, 'Acuracia', 'Comparação Acurácia - Classificação')
plot_metric_comparison(classification_results, 'F1-score', 'Comparação F1-score - Classificação')
plot_metric_comparison(classification_results, 'Kappa', 'Comparação Kappa - Classificação')

## Justificativas e Análise Crítica
Os resultados obtidos mostram diferenças relevantes entre os algoritmos utilizados para cada tarefa. São destacados pontos como desempenho, robustez, interpretabilidade e importância do pré-processamento.

A realização de múltiplos experimentos e repetições garante maior confiabilidade estatística dos resultados.